# Baseline decoders

Two very simple decoders, to see how the data, the score and a submission fit together. Neither one uses any quantum error correction theory.

Each shot gives 70 **detector events** (D0–D69, which checks fired) and the decoder must predict 4 **observable flips** (L0–L3, whether each logical qubit was flipped). The score is the **logical error rate (LER)**: the fraction of predicted L bits that are wrong. Lower is better.

Run from `notebooks/tutorials/`.

In [1]:
import numpy as np
from hackathon_2026 import load_kaggle_csv, logical_error_rate, save_submission

train = load_kaggle_csv("../../data/train.csv")     # has labels L0-L3
test = load_kaggle_csv("../../data/test_set.csv")   # no labels: this is what you submit

print("detector events:", train["detector_events"].shape, "| observable flips:", train["observable_flips"].shape)
print("first shot:", train["id"][0], train["basis"][0])
print("  D:", train["detector_events"][0].astype(int))
print("  L:", train["observable_flips"][0].astype(int))

detector events: (200000, 70) | observable flips: (200000, 4)
first shot: gen_X_00000 X
  D: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
  L: [0 0 0 0]


## Hold out part of the training data

To score a decoder honestly, test it on shots it did not learn from. Split the labelled shots in half: learn from `fit`, score on `val`.

In [2]:
rng = np.random.default_rng(0)
order = rng.permutation(len(train["id"]))
fit_idx, val_idx = order[: len(order) // 2], order[len(order) // 2 :]

def subset(d, idx):
    return {k: v[idx] for k, v in d.items()}

fit, val = subset(train, fit_idx), subset(train, val_idx)
print(len(fit["id"]), "shots to learn from,", len(val["id"]), "shots to score on")

100000 shots to learn from, 100000 shots to score on


## Baseline 1: always predict "no flip"

Ignore the detectors completely and predict 0 for every L. Its LER is simply how often each logical qubit really flips.

In [4]:
def decode_zeros(basis, detector_events):
    return np.zeros((len(detector_events), 4), dtype=np.uint8)

pred = decode_zeros(val["basis"], val["detector_events"])
print("LER, always 0:", logical_error_rate(pred, val["observable_flips"]))
print("how often each L flips:", val["observable_flips"].mean(axis=0).round(3))

LER, always 0: 0.28813
how often each L flips: [0.309 0.322 0.24  0.281]


## Baseline 2: a lookup table

Memorise the training data. For every detector pattern seen in `fit`, remember which L flips came with it most often. To decode, look the pattern up; if it was never seen, fall back to "no flip".

There are $2^{70}$ possible patterns, so many shots in `val` will have a pattern that never appeared in `fit`. That is exactly why real decoders use the structure of the circuit instead.

In [5]:
from collections import Counter, defaultdict

def key(basis, row):
    return basis + np.packbits(row).tobytes().hex()   # one string per (basis, detector pattern)

def build_table(data):
    counts = defaultdict(Counter)
    for b, dets, obs in zip(data["basis"], data["detector_events"], data["observable_flips"]):
        counts[key(b, dets)][tuple(obs.astype(int))] += 1
    return {k: c.most_common(1)[0][0] for k, c in counts.items()}

def decode_lookup(table, basis, detector_events):
    pred = np.zeros((len(detector_events), 4), dtype=np.uint8)
    for i, (b, dets) in enumerate(zip(basis, detector_events)):
        pred[i] = table.get(key(b, dets), (0, 0, 0, 0))
    return pred

table = build_table(fit)
seen = np.mean([key(b, d) in table for b, d in zip(val["basis"], val["detector_events"])])
pred = decode_lookup(table, val["basis"], val["detector_events"])
print(f"{len(table)} distinct patterns memorised; {seen:.1%} of val shots have a pattern seen before")
print("LER, lookup table:", logical_error_rate(pred, val["observable_flips"]))

81162 distinct patterns memorised; 21.9% of val shots have a pattern seen before
LER, lookup table: 0.2649975


## Write a submission

Rebuild the table from all labelled shots, decode the test set, and save the file to upload to Kaggle.

In [ ]:
table = build_table(train)
pred = decode_lookup(table, test["basis"], test["detector_events"])
save_submission("../../submission_baseline.csv", test["id"], pred)
print(open("../../submission_baseline.csv").read()[:120])